# 01 StatsBomb-Daten laden

Dieses Notebook implementiert BD-05. Es laedt StatsBomb Open Data fuer den in BD-04 definierten Turnier-Scope, speichert Match- und Eventdaten und fuehrt erste technische Qualitaetschecks aus.

Outputs:

- `data/raw/statsbomb/matches_raw.parquet`
- `data/raw/statsbomb/events_raw.parquet/` als Parquet-Dataset mit Part-Dateien
- `data/bronze/statsbomb_matches.parquet`
- `data/bronze/statsbomb_events.parquet/` als Parquet-Dataset mit Part-Dateien

## Methodischer Ansatz

Die externen JSON-Dateien werden reproduzierbar geladen, in strukturierte Tabellen ueberfuehrt und nach Pipeline-Stufe gespeichert. Rohdaten bleiben in `data/raw/` erhalten; technisch normalisierte Tabellen werden in `data/bronze/` abgelegt.

Die Eventdaten werden als Parquet-Dataset geschrieben, damit grosse Match-Event-Mengen effizient gelesen und in nachfolgenden Feature-Schritten verarbeitet werden koennen.

In [ ]:
import json
import time
import shutil

import pandas as pd
import requests

from project_paths import project_root as get_project_root
from pipeline_config import STATSBOMB_BASE_URL, STATSBOMB_COMPETITION_ALIASES

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 20)

base_path = get_project_root()
docs_path = base_path / 'docs'
raw_path = base_path / 'data' / 'raw' / 'statsbomb'
bronze_path = base_path / 'data' / 'bronze'
cache_path = base_path / 'data' / 'cache' / 'statsbomb_open_data'

for path in [raw_path, bronze_path, cache_path]:
    path.mkdir(parents=True, exist_ok=True)

scope_path = base_path / 'data' / 'reference' / 'tournament_scope.json'
with scope_path.open('r', encoding='utf-8') as f:
    scope = json.load(f)

scope_tournaments = pd.DataFrame(scope['tournaments'])
scope_tournaments['season_name'] = scope_tournaments['season_name'].astype(str)

# StatsBomb nutzt teilweise andere offizielle Wettbewerbsnamen als unser Projekt-Scope.
scope_tournaments['statsbomb_competition_name'] = scope_tournaments['competition_name'].replace(STATSBOMB_COMPETITION_ALIASES)

{
    'base_path': str(base_path),
    'scope_path': str(scope_path),
    'tournaments_in_scope': len(scope_tournaments),
    'expected_matches': int(scope['expected_totals']['matches']),
}


## StatsBomb Open Data laden

StatsBomb Open Data liegt als JSON-Dateistruktur vor. Das Notebook nutzt direkt das oeffentliche GitHub-Repository. Kleine Metadaten- und Match-Dateien werden in `data/cache/statsbomb_open_data/` abgelegt; die grossen Event-Dateien werden bewusst nicht gecached, damit der lokale Speicher nicht unnoetig volllaeuft.

In [ ]:
from pathlib import Path
def cache_file_for(relative_path: str) -> Path:
    return cache_path / relative_path


def load_statsbomb_json(relative_path: str, use_cache: bool = True):
    cache_file = cache_file_for(relative_path)
    if use_cache and cache_file.exists():
        with cache_file.open('r', encoding='utf-8') as f:
            return json.load(f)

    url = f'{STATSBOMB_BASE_URL}/{relative_path}'
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    data = response.json()

    if use_cache:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        temp_cache_file = cache_file.with_suffix(cache_file.suffix + '.tmp')
        with temp_cache_file.open('w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)
        temp_cache_file.replace(cache_file)

    time.sleep(0.05)
    return data


def make_parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    object_columns = result.select_dtypes(include=['object']).columns
    for column in object_columns:
        contains_nested_values = result[column].map(lambda value: isinstance(value, (dict, list))).any()
        if contains_nested_values:
            result[column] = result[column].map(
                lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
            )
    return result


competitions = pd.json_normalize(load_statsbomb_json('competitions.json'), sep='_')
competitions['season_name'] = competitions['season_name'].astype(str)

competitions[['competition_id', 'season_id', 'competition_name', 'season_name']].head()

## Turnier-Scope anwenden

Der fachliche Filter kommt aus `data/reference/tournament_scope.json`. Dadurch verwenden alle Pipeline-Schritte dieselbe Turnierliste.

In [ ]:
competitions_for_join = competitions.rename(columns={'competition_name': 'statsbomb_competition_name'})[
    ['competition_id', 'season_id', 'statsbomb_competition_name', 'season_name']
]

selected_competitions = scope_tournaments.merge(
    competitions_for_join,
    on=['statsbomb_competition_name', 'season_name'],
    how='left',
)

missing_competitions = selected_competitions[selected_competitions['competition_id'].isna()]
if not missing_competitions.empty:
    raise ValueError(
        'Nicht alle Turniere aus data/reference/tournament_scope.json wurden in StatsBomb competitions.json gefunden: '
        + missing_competitions[['scope_id', 'competition_name', 'statsbomb_competition_name', 'season_name']].to_dict('records').__repr__()
    )

selected_competitions = selected_competitions.astype({
    'competition_id': 'int64',
    'season_id': 'int64',
})

selected_competitions[['scope_id', 'short_name', 'competition_name', 'statsbomb_competition_name', 'season_name', 'competition_id', 'season_id', 'expected_matches']]

## Matchdaten laden und speichern

In [ ]:
match_frames = []

for tournament in selected_competitions.to_dict('records'):
    relative_path = f"matches/{tournament['competition_id']}/{tournament['season_id']}.json"
    tournament_matches = pd.json_normalize(load_statsbomb_json(relative_path), sep='_')

    tournament_matches['scope_id'] = tournament['scope_id']
    tournament_matches['short_name'] = tournament['short_name']
    tournament_matches['competition_id'] = tournament['competition_id']
    tournament_matches['season_id'] = tournament['season_id']
    tournament_matches['competition_name'] = tournament['competition_name']
    tournament_matches['statsbomb_competition_name'] = tournament['statsbomb_competition_name']
    tournament_matches['season_name'] = tournament['season_name']
    tournament_matches['expected_matches'] = tournament['expected_matches']

    match_frames.append(tournament_matches)

matches_raw = pd.concat(match_frames, ignore_index=True)
matches_raw['match_id'] = matches_raw['match_id'].astype('int64')
matches_raw = make_parquet_safe(matches_raw)

matches_raw.to_parquet(raw_path / 'matches_raw.parquet', index=False)
matches_raw.to_parquet(bronze_path / 'statsbomb_matches.parquet', index=False)

{
    'matches': len(matches_raw),
    'unique_match_ids': matches_raw['match_id'].nunique(),
    'raw_output': str(raw_path / 'matches_raw.parquet'),
    'bronze_output': str(bronze_path / 'statsbomb_matches.parquet'),
}

## Eventdaten pro Match laden und speichern

Die Eventdaten werden pro `match_id` geladen und sofort als Parquet-Part geschrieben. Dadurch muss das Notebook nicht alle Events gleichzeitig im RAM halten. Wichtige Scope-Felder werden an jede Eventzeile angehaengt, damit Feature-Notebooks nicht erneut Competition-/Season-Metadaten joinen muessen.

In [ ]:
events_raw_output = raw_path / 'events_raw.parquet'
events_bronze_output = bronze_path / 'statsbomb_events.parquet'

for output_path in [events_raw_output, events_bronze_output]:
    if output_path.exists():
        if output_path.is_dir():
            shutil.rmtree(output_path)
        else:
            output_path.unlink()
    output_path.mkdir(parents=True, exist_ok=True)

event_manifest_rows = []

match_context_columns = [
    'match_id',
    'scope_id',
    'short_name',
    'competition_id',
    'season_id',
    'competition_name',
    'statsbomb_competition_name',
    'season_name',
    'match_date',
    'home_team_home_team_name',
    'away_team_away_team_name',
    'stadium_name',
]

available_context_columns = [column for column in match_context_columns if column in matches_raw.columns]

for idx, match in enumerate(matches_raw[available_context_columns].to_dict('records'), start=1):
    match_id = int(match['match_id'])
    relative_path = f'events/{match_id}.json'
    events = pd.json_normalize(load_statsbomb_json(relative_path, use_cache=False), sep='_')

    if events.empty:
        events = pd.DataFrame([{'match_id': match_id}])
    else:
        events['match_id'] = match_id

    for column, value in match.items():
        if column != 'match_id':
            events[column] = value

    events['match_id'] = events['match_id'].astype('int64')
    events = make_parquet_safe(events)

    part_file = f'part-{idx:06d}-match-{match_id}.parquet'
    events.to_parquet(events_raw_output / part_file, index=False)
    events.to_parquet(events_bronze_output / part_file, index=False)

    event_manifest_rows.append({
        'match_id': match_id,
        'scope_id': match.get('scope_id'),
        'events': int(len(events)),
        'raw_part': str(events_raw_output / part_file),
        'bronze_part': str(events_bronze_output / part_file),
    })

    del events

    if idx % 25 == 0 or idx == len(matches_raw):
        print(f'{idx}/{len(matches_raw)} matches geladen')

events_manifest = pd.DataFrame(event_manifest_rows)
events_manifest.to_parquet(raw_path / 'events_manifest.parquet', index=False)

{
    'events': int(events_manifest['events'].sum()),
    'matches_with_events': int(events_manifest['match_id'].nunique()),
    'raw_output_dir': str(events_raw_output),
    'bronze_output_dir': str(events_bronze_output),
    'manifest_output': str(raw_path / 'events_manifest.parquet'),
}

## Erste Datenqualitaetschecks

In [ ]:
match_counts = (
    matches_raw.groupby(['scope_id', 'short_name', 'competition_name', 'season_name'], as_index=False)
    .agg(
        actual_matches=('match_id', 'nunique'),
        expected_matches=('expected_matches', 'first'),
    )
)
match_counts['matches_ok'] = match_counts['actual_matches'] == match_counts['expected_matches']

events_per_tournament = (
    events_manifest.groupby(['scope_id'], as_index=False)
    .agg(
        events=('events', 'sum'),
        matches_with_events=('match_id', 'nunique'),
    )
)

quality_summary = match_counts.merge(events_per_tournament, on='scope_id', how='left')
quality_summary

In [ ]:
match_ids = set(matches_raw['match_id'])
event_match_ids = set(events_manifest['match_id'])

missing_event_match_ids = sorted(match_ids - event_match_ids)
unexpected_event_match_ids = sorted(event_match_ids - match_ids)

team_columns = [
    column
    for column in ['home_team_home_team_name', 'away_team_away_team_name']
    if column in matches_raw.columns
]
missing_team_names = int(matches_raw[team_columns].isna().sum().sum()) if team_columns else None

dq_checks = {
    'expected_matches': int(scope['expected_totals']['matches']),
    'actual_matches': int(matches_raw['match_id'].nunique()),
    'events': int(events_manifest['events'].sum()),
    'missing_event_match_ids': missing_event_match_ids,
    'unexpected_event_match_ids': unexpected_event_match_ids,
    'missing_team_names': missing_team_names,
    'duplicate_match_rows': int(matches_raw.duplicated(subset=['match_id']).sum()),
}

assert dq_checks['actual_matches'] == dq_checks['expected_matches']
assert not missing_event_match_ids
assert not unexpected_event_match_ids
assert dq_checks['missing_team_names'] == 0
assert dq_checks['duplicate_match_rows'] == 0

dq_checks

## Ergebnis

Wenn alle Assertions erfolgreich sind, sind die StatsBomb Match- und Eventdaten fuer den definierten Turnier-Scope gespeichert. Nachfolgende Notebooks koennen aus `data/bronze/statsbomb_matches.parquet` und `data/bronze/statsbomb_events.parquet` lesen.